In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("HotelBigDataSQL") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("hdfs://localhost:9000/user/hotel/data/data_cleaned_hotel.csv")

df.createOrReplaceTempView("hotel")
print("Đã nạp dữ liệu thành công! Sẵn sàng chạy các câu truy vấn SQL.")

# CÂU 7: Khảo sát tương quan giữa yêu cầu đặc biệt và khả năng hủy phòng
query_7 = """
SELECT 
    total_of_special_requests,
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS total_canceled,
    ROUND(SUM(is_canceled) / COUNT(*) * 100, 2) AS cancel_rate_pct
FROM hotel
GROUP BY total_of_special_requests
ORDER BY total_of_special_requests ASC
"""

print("\nCÂU 7: TƯƠNG QUAN GIỮA YÊU CẦU ĐẶC BIỆT VÀ TỶ LỆ HỦY PHÒNG")
result_7 = spark.sql(query_7)
result_7.show(truncate=False)

# CÂU 8: Tác động của chính sách Đặt cọc đến quyết định hủy phòng
query_8 = """
WITH deposit_stats AS (
    SELECT 
        deposit_type,
        COUNT(*) AS total_bookings,
        SUM(is_canceled) AS total_canceled,
        ROUND(SUM(is_canceled) / COUNT(*) * 100, 2) AS cancel_rate_pct
    FROM hotel
    GROUP BY deposit_type
)
SELECT 
    deposit_type,
    total_bookings,
    total_canceled,
    cancel_rate_pct,
    CASE 
        WHEN cancel_rate_pct > 50 THEN 'Nguy co cao'
        WHEN cancel_rate_pct >= 20 AND cancel_rate_pct <= 50 THEN 'Trung binh'
        ELSE 'An toan'
    END AS risk_level
FROM deposit_stats
ORDER BY cancel_rate_pct DESC
"""

print("\nCÂU 8: TÁC ĐỘNG CỦA CHÍNH SÁCH ĐẶT CỌC ĐẾN HỦY PHÒNG")
result_8 = spark.sql(query_8)
result_8.show(truncate=False)

Đã nạp dữ liệu thành công! Sẵn sàng chạy các câu truy vấn SQL.

CÂU 7: TƯƠNG QUAN GIỮA YÊU CẦU ĐẶC BIỆT VÀ TỶ LỆ HỦY PHÒNG
+-------------------------+--------------+--------------+---------------+
|total_of_special_requests|total_bookings|total_canceled|cancel_rate_pct|
+-------------------------+--------------+--------------+---------------+
|0                        |70318         |33556         |47.72          |
|1                        |33226         |7318          |22.02          |
|2                        |12969         |2866          |22.1           |
|3                        |2497          |446           |17.86          |
|4                        |340           |36            |10.59          |
|5                        |40            |2             |5.0            |
+-------------------------+--------------+--------------+---------------+


CÂU 8: TÁC ĐỘNG CỦA CHÍNH SÁCH ĐẶT CỌC ĐẾN HỦY PHÒNG
+------------+--------------+--------------+---------------+-----------+
|deposit_